# SD3.5 LoRA Fine-Tuning – CityPersons Pedestrian Insertion

**Goal:** Fine-tune Stable Diffusion 3.5 Medium with LoRA on the CityPersons dataset so that the model learns the CityPersons camera viewpoint, perspective, street-scene geometry, and realistic pedestrian anatomy.  
The trained LoRA is then loaded by the augmentation notebook (`citypersons_sd35_lora_kaggle.ipynb`) to improve the quality of pedestrian-insertion img2img augmentations used as training data for downstream pedestrian detectors.

## Pipeline Overview
```
1. Install & verify dependencies
2. Download diffusers training script (train_dreambooth_lora_sd3.py)
3. HuggingFace login (gated model)
4. Configuration – all hyperparams in one cell
5. Prepare training dataset folder (images + per-image captions)
6. Preview dataset
7. Run LoRA training (accelerate launch)
8. Monitor training loss curve
9. Sanity-check inference with trained LoRA
10. Package & save LoRA weights
```

## Hardware
- Kaggle T4 ×2 (16 GB VRAM each)
- Training runs on **cuda:0** only; cuda:1 is used for inference smoke-test.

## Dataset
- **CityPersons** – Kaggle dataset `samyamine23/cityperson`  
  Path inside Kaggle: `/kaggle/input/cityperson` (or `/kaggle/input/citypersons`)

## References
- [Stability AI SD3.5 Fine-Tuning Guide](https://stabilityai.notion.site/Stable-Diffusion-3-5-fine-tuning-guide-11a61cdcd1968027a15bdbd7c40be8c6)
- [diffusers train_dreambooth_lora_sd3.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/train_dreambooth_lora_sd3.py)


## 1. Install Dependencies

We pin versions carefully to avoid CUDA / RAPIDS conflicts on Kaggle.  
**Do NOT reinstall `torch`, `torchvision`, or `xformers`.** Kaggle's pre-installed CUDA stack must stay intact.

In [ ]:
%%bash
pip install -q \
  "diffusers>=0.31.0,<1.0.0" \
  "transformers>=4.44.0" \
  "peft>=0.12.0" \
  "accelerate>=0.33.0" \
  "bitsandbytes>=0.43.0" \
  "datasets>=2.20.0" \
  sentencepiece protobuf safetensors Pillow tqdm

## 2. Download diffusers Training Script

We use the official `train_dreambooth_lora_sd3.py` from the HuggingFace diffusers repository.  
This script supports SD3/SD3.5 with Flow Matching loss, mixed precision, gradient checkpointing, and 8-bit Adam.

In [ ]:
import os
import urllib.request

SCRIPT_URL = (
    "https://raw.githubusercontent.com/huggingface/diffusers/main/"
    "examples/dreambooth/train_dreambooth_lora_sd3.py"
)
SCRIPT_PATH = "/kaggle/working/train_dreambooth_lora_sd3.py"

if not os.path.exists(SCRIPT_PATH):
    print("Downloading train_dreambooth_lora_sd3.py ...")
    urllib.request.urlretrieve(SCRIPT_URL, SCRIPT_PATH)
    print("Downloaded to", SCRIPT_PATH)
else:
    print("Script already present:", SCRIPT_PATH)

# Also download requirements file
REQS_URL = (
    "https://raw.githubusercontent.com/huggingface/diffusers/main/"
    "examples/dreambooth/requirements_sd3.txt"
)
REQS_PATH = "/kaggle/working/requirements_sd3.txt"
try:
    urllib.request.urlretrieve(REQS_URL, REQS_PATH)
    print("Downloaded requirements_sd3.txt")
except Exception as exc:
    print("Could not download requirements_sd3.txt (non-critical):", exc)

## 3. Runtime Check

In [ ]:
import torch
import diffusers
import transformers
import peft
import accelerate

print("torch         :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda runtime  :", torch.version.cuda)
    print("gpu count     :", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f"  gpu {idx}: {props.name}  VRAM={props.total_memory / 1e9:.1f} GB")
print("diffusers     :", diffusers.__version__)
print("transformers  :", transformers.__version__)
print("peft          :", peft.__version__)
print("accelerate    :", accelerate.__version__)

## 4. HuggingFace Login

SD3.5 Medium is a **gated** model. You need to:
1. Accept the model license on [huggingface.co/stabilityai/stable-diffusion-3.5-medium](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium)
2. Add a Kaggle secret called `HF_TOKEN` with your HuggingFace access token.

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to HuggingFace with Kaggle secret HF_TOKEN.")
except Exception as exc:
    print("HF login skipped / failed. Make sure HF_TOKEN Kaggle secret is set.")
    print(type(exc).__name__, exc)

## 5. Configuration

All hyperparameters are here. Adjust before running training.

| Parameter | Default | Notes |
|---|---|---|
| `RESOLUTION` | 512 | Lower = less VRAM. 768 is possible but tight on T4 |
| `LORA_RANK` | 16 | 8 if OOM, 32 for higher quality |
| `MAX_TRAIN_STEPS` | 1000 | ~500–2000 for street scenes |
| `LEARNING_RATE` | 1e-4 | Typical for LoRA on SD3.5 |
| `TRAIN_BATCH_SIZE` | 1 | Must be 1 on T4 |
| `GRADIENT_ACCUM_STEPS` | 4 | Effective batch = 4 |
| `USE_T5` | False | T5-XXL needs extra 10 GB; skip on T4 |
| `MAX_TRAIN_IMAGES` | 500 | Cap for faster experiments |


In [ ]:
from pathlib import Path

# ── Dataset ────────────────────────────────────────────────────────────────────
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/samyamine23/cityperson"),
    Path("/kaggle/input/cityperson"),
    Path("/kaggle/input/citypersons"),
    Path("/kaggle/input/city-persons"),
]
DATASET_ROOT = next(
    (p for p in DATASET_ROOT_CANDIDATES if p.exists()),
    DATASET_ROOT_CANDIDATES[1],  # fallback
)

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# ── Output ─────────────────────────────────────────────────────────────────────
TRAIN_DATA_DIR  = Path("/kaggle/working/lora_train_data")   # prepared dataset
OUTPUT_DIR      = Path("/kaggle/working/sd35-lora-output")   # LoRA weights
LOG_DIR         = Path("/kaggle/working/training_logs")
TRAIN_LOG_CSV   = LOG_DIR / "train_log.csv"
LOSS_CURVE_PNG  = LOG_DIR / "loss_curve.png"
INFERENCE_DIR   = Path("/kaggle/working/lora_sanity_check")

for d in [TRAIN_DATA_DIR, OUTPUT_DIR, LOG_DIR, INFERENCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ───────────────────────────────────────────────────
RESOLUTION           = 512       # image resolution for training
LORA_RANK            = 16        # LoRA rank; use 8 if OOM
LORA_ALPHA           = 16        # LoRA alpha (usually = rank)
MAX_TRAIN_IMAGES     = 500       # cap training images; None = all
MAX_TRAIN_STEPS      = 1000      # total gradient steps
LEARNING_RATE        = 1e-4
TRAIN_BATCH_SIZE     = 1
GRADIENT_ACCUM_STEPS = 4         # effective batch = 4
MIXED_PRECISION      = "fp16"    # fp16 for T4 (no BF16 support)
USE_T5               = False     # skip T5-XXL on T4 to save VRAM
SAVE_STEPS           = 250       # checkpoint every N steps
SEED                 = 42

# ── Prompts ────────────────────────────────────────────────────────────────────
# Instance prompt: describes WHAT is in the training images
INSTANCE_PROMPT = (
    "a photorealistic CityPersons urban street scene photograph, "
    "dashcam perspective, pedestrians on sidewalk and roadside, "
    "European city, correct perspective, natural lighting"
)

# Caption template per augmentation variant (used during inference)
VARIANT_PROMPTS = {
    "add_single_pedestrian":   "add one realistic full-body pedestrian naturally standing or walking on the sidewalk or roadside, correct scale, correct perspective, natural contact with ground",
    "add_two_pedestrians":     "add two realistic pedestrians walking naturally in the urban scene, correct scale and perspective, preserve road layout and existing objects",
    "add_small_group":         "add a small realistic group of pedestrians in a plausible sidewalk area, natural spacing, correct perspective, surveillance camera realism",
    "add_occluded_pedestrian": "add one realistic partially occluded pedestrian behind an existing object or near other pedestrians, plausible occlusion, correct scale",
    "add_distant_pedestrian":  "add one small distant pedestrian far along the street, correct perspective scaling, natural pose, preserve scene geometry",
    "add_near_pedestrian":     "add one larger foreground pedestrian near the lower image region, realistic anatomy, correct ground contact, preserve camera viewpoint",
}

print(f"DATASET_ROOT    : {DATASET_ROOT}")
print(f"MODEL_ID        : {MODEL_ID}")
print(f"RESOLUTION      : {RESOLUTION}")
print(f"LORA_RANK       : {LORA_RANK}")
print(f"MAX_TRAIN_STEPS : {MAX_TRAIN_STEPS}")
print(f"LEARNING_RATE   : {LEARNING_RATE}")
print(f"MIXED_PRECISION : {MIXED_PRECISION}")
print(f"USE_T5          : {USE_T5}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

## 6. Prepare Training Dataset

The diffusers DreamBooth script expects:
```
lora_train_data/
  image_001.jpg   ← training image (JPEG or PNG)
  image_001.txt   ← matching caption (same stem, .txt extension)
  image_002.jpg
  image_002.txt
  ...
```
When `--caption_column` is omitted, the script reads `<stem>.txt` next to each image.

We resize images to `RESOLUTION × RESOLUTION` (center-crop) and write a per-image caption file.

In [ ]:
import csv
import math
import random
import shutil
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

from PIL import Image, ImageOps

warnings.filterwarnings("ignore", category=UserWarning)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_source_image(path: Path) -> bool:
    if path.suffix.lower() not in IMAGE_EXTS:
        return False
    name = path.name.lower()
    if any(tok in name for tok in ["mask", "label", "gtfine", "gtbbox",
                                    "instance", "polygon", "color"]):
        return False
    return True


def collect_image_paths(root: Path, max_images: Optional[int] = None):
    """Recursively collect source image paths from root."""
    paths = []
    for p in sorted(root.rglob("*")):
        if p.is_file() and is_source_image(p):
            paths.append(p)
            if max_images and len(paths) >= max_images:
                break
    return paths


def center_crop_resize(image: Image.Image, size: int) -> Image.Image:
    """Resize shortest side to `size`, then center-crop to size×size."""
    w, h = image.size
    scale = size / min(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    image = image.resize((new_w, new_h), Image.LANCZOS)
    left  = (new_w - size) // 2
    top   = (new_h - size) // 2
    return image.crop((left, top, left + size, top + size))


def build_caption(image_path: Path) -> str:
    """Build a descriptive caption for a CityPersons image."""
    # Try to infer split and city from path parts
    parts_lower = [p.lower() for p in image_path.parts]
    split = next((s for s in ["train", "val", "test"] if s in parts_lower), "train")

    # City is often encoded in the filename: e.g. aachen_000014_000019_leftImg8bit.png
    city = image_path.stem.split("_")[0] if "_" in image_path.stem else "city"

    caption = (
        f"a photorealistic CityPersons urban street scene photograph of {city}, "
        "dashcam or surveillance camera perspective, European city environment, "
        "pedestrians on sidewalk and roadside, vehicles on road, "
        "correct perspective and scale, natural daylight or artificial lighting, "
        "high fidelity, preserve background geometry and road layout"
    )
    return caption


def prepare_train_data(
    dataset_root: Path,
    output_dir: Path,
    resolution: int = 512,
    max_images: Optional[int] = None,
    overwrite: bool = False,
):
    """
    Copy CityPersons images into output_dir, resized to resolution×resolution,
    and write a matching caption .txt file for each.
    """
    image_paths = collect_image_paths(dataset_root, max_images=max_images)
    if not image_paths:
        raise FileNotFoundError(
            f"No images found under {dataset_root}. "
            "Check that the CityPersons Kaggle dataset is mounted."
        )

    print(f"Found {len(image_paths)} source images under {dataset_root}")

    prepared = 0
    skipped  = 0
    for src in image_paths:
        stem    = src.stem
        dst_img = output_dir / f"{stem}.jpg"
        dst_txt = output_dir / f"{stem}.txt"

        if dst_img.exists() and dst_txt.exists() and not overwrite:
            skipped += 1
            continue

        try:
            img = Image.open(src).convert("RGB")
            img = center_crop_resize(img, resolution)
            img.save(dst_img, "JPEG", quality=95)
            dst_txt.write_text(build_caption(src), encoding="utf-8")
            prepared += 1
        except Exception as exc:
            print(f"  SKIP {src.name}: {exc}")

    total = prepared + skipped
    print(f"Prepared  : {prepared} images (new/overwritten)")
    print(f"Skipped   : {skipped} images (already exist)")
    print(f"Total     : {total} training images in {output_dir}")
    return total


# ── Run preparation ────────────────────────────────────────────────────────────
if DATASET_ROOT.exists():
    n_prepared = prepare_train_data(
        dataset_root=DATASET_ROOT,
        output_dir=TRAIN_DATA_DIR,
        resolution=RESOLUTION,
        max_images=MAX_TRAIN_IMAGES,
        overwrite=False,
    )
    print(f"\nTraining data ready: {TRAIN_DATA_DIR}")
else:
    print(f"WARNING: Dataset root not found: {DATASET_ROOT}")
    print("Check that CityPersons dataset is added as a Kaggle dataset source.")
    n_prepared = 0

## 7. Dataset Preview

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

train_images = sorted(TRAIN_DATA_DIR.glob("*.jpg"))
print(f"Total training images in {TRAIN_DATA_DIR}: {len(train_images)}")

# Preview first 6 images + their captions
sample = train_images[:6]
if sample:
    cols = min(3, len(sample))
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax, img_path in zip(axes, sample):
        img = Image.open(img_path)
        caption_path = img_path.with_suffix(".txt")
        caption = caption_path.read_text(encoding="utf-8") if caption_path.exists() else ""
        ax.imshow(img)
        ax.set_title(img_path.name[:30], fontsize=8)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.suptitle("Training Images Sample", fontsize=12)
    plt.tight_layout()
    plt.show()

    # Print caption of first image
    cap_file = sample[0].with_suffix(".txt")
    if cap_file.exists():
        print("\nCaption sample for", sample[0].name, ":")
        print(cap_file.read_text(encoding="utf-8"))
else:
    print("No images found in training data dir. Run cell 6 first.")

## 8. Write `accelerate` Config

We write a minimal accelerate config so we can call `accelerate launch` without interactive setup.  
Single GPU (cuda:0) config – T4 does not support BF16, so we use FP16.

In [ ]:
import os
from pathlib import Path

ACCEL_CONFIG_DIR  = Path("/root/.cache/huggingface/accelerate")
ACCEL_CONFIG_PATH = ACCEL_CONFIG_DIR / "default_config.yaml"
ACCEL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

accel_config = """\
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

ACCEL_CONFIG_PATH.write_text(accel_config)
print("Wrote accelerate config to", ACCEL_CONFIG_PATH)

## 9. Build Training Command

We build the `accelerate launch` command string here so it is easy to inspect and tweak before running.

### Key flags
| Flag | Value | Why |
|---|---|---|
| `--mixed_precision` | fp16 | Halves VRAM on T4 |
| `--gradient_checkpointing` | (set) | Trades compute for memory |
| `--use_8bit_adam` | (set) | 8-bit AdamW via bitsandbytes |
| `--train_text_encoder` | NOT set | Freeze text encoders to save 6 GB VRAM |
| `--rank` | 16 | LoRA rank |
| `--with_prior_preservation` | NOT set | No prior – keeps VRAM low |
| `--resume_from_checkpoint` | latest | Auto-resume if run crashes |

In [ ]:
import shlex

TRAIN_SCRIPT = "/kaggle/working/train_dreambooth_lora_sd3.py"

def build_training_command() -> str:
    args = [
        "accelerate", "launch",
        "--mixed_precision", MIXED_PRECISION,
        TRAIN_SCRIPT,
        # Model
        "--pretrained_model_name_or_path", MODEL_ID,
        # Dataset
        "--instance_data_dir", str(TRAIN_DATA_DIR),
        "--instance_prompt", INSTANCE_PROMPT,
        # Output
        "--output_dir", str(OUTPUT_DIR),
        # Resolution & batch
        "--resolution", str(RESOLUTION),
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUM_STEPS),
        # Training schedule
        "--max_train_steps", str(MAX_TRAIN_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--lr_scheduler", "cosine",
        "--lr_warmup_steps", str(int(MAX_TRAIN_STEPS * 0.05)),
        # LoRA
        "--rank", str(LORA_RANK),
        # Memory optimisations
        "--gradient_checkpointing",
        "--use_8bit_adam",
        # Checkpointing
        "--checkpointing_steps", str(SAVE_STEPS),
        "--resume_from_checkpoint", "latest",
        # Misc
        "--seed", str(SEED),
        "--report_to", "none",
    ]

    # Skip T5 text encoder to save ~6 GB VRAM on T4
    if not USE_T5:
        # The script checks huggingface model config; we skip T5 by
        # not passing --train_text_encoder_3 and noting it in instance prompt.
        # Some diffusers versions expose --max_sequence_length for T5 control.
        # Setting it to 0 effectively disables T5 embeddings.
        args += ["--max_sequence_length", "0"]

    return " ".join(shlex.quote(a) for a in args)


TRAIN_CMD = build_training_command()
print("Training command:")
print(TRAIN_CMD)
print()
print(f"Total images : {n_prepared}")
print(f"Steps/epoch  : {max(1, n_prepared // TRAIN_BATCH_SIZE)}")
print(f"Total epochs : ~{MAX_TRAIN_STEPS / max(1, n_prepared // TRAIN_BATCH_SIZE):.1f}")

## 10. Run LoRA Training

> **Estimated time on Kaggle T4:**  
> - 500 steps @ resolution 512 ≈ 25–35 min  
> - 1000 steps @ resolution 512 ≈ 50–70 min  

Training logs are streamed to stdout below.  
Intermediate checkpoints are saved to `OUTPUT_DIR/checkpoint-<step>/` every `SAVE_STEPS` steps  
so you can resume if the Kaggle session times out.

In [ ]:
import subprocess
import sys
import time

if n_prepared == 0:
    raise RuntimeError(
        "No training images prepared. "
        "Check that the CityPersons dataset is mounted and run Section 6 first."
    )

print("Starting LoRA training ...")
print("Command:", TRAIN_CMD)
print("-" * 80)

t0 = time.time()
result = subprocess.run(
    TRAIN_CMD,
    shell=True,
    stdout=sys.stdout,
    stderr=sys.stderr,
    text=True,
)
elapsed = time.time() - t0

print("-" * 80)
if result.returncode == 0:
    print(f"Training completed successfully in {elapsed/60:.1f} min.")
    print(f"LoRA weights saved to: {OUTPUT_DIR}")
else:
    print(f"Training exited with code {result.returncode} after {elapsed/60:.1f} min.")
    print("Check output above for error messages.")

### 10b. (Optional) Manual Custom Training Loop

If `train_dreambooth_lora_sd3.py` fails (e.g., flag incompatibility with your diffusers version),  
the cell below provides a **self-contained LoRA training loop** using the same libraries.  
It matches the approach in the augmentation notebook but with:
- proper Flow Matching loss (not MSE on timestep-level noise)
- gradient checkpointing
- 8-bit Adam
- per-image captions from `.txt` files
- checkpoint saving

In [ ]:
# ── Run this cell ONLY if Section 10 fails. ──────────────────────────────────
# Set RUN_MANUAL_LOOP = True to activate.
RUN_MANUAL_LOOP = False  # <-- change to True if needed

if RUN_MANUAL_LOOP:
    import gc
    import math
    import time
    import warnings
    import csv as _csv

    import torch
    import torch.nn.functional as F
    import bitsandbytes as bnb
    from PIL import Image
    from peft import LoraConfig, get_peft_model
    from diffusers import StableDiffusion3Pipeline
    from diffusers.models.attention_processor import AttnProcessor2_0

    warnings.filterwarnings("ignore", category=FutureWarning)

    DEVICE = "cuda:0"

    # ── Dataset ──────────────────────────────────────────────────────────────
    def load_train_pairs(data_dir: Path, max_items: Optional[int] = None):
        pairs = []
        for img_path in sorted(data_dir.glob("*.jpg")):
            txt_path = img_path.with_suffix(".txt")
            caption = (
                txt_path.read_text(encoding="utf-8").strip()
                if txt_path.exists()
                else INSTANCE_PROMPT
            )
            pairs.append((img_path, caption))
            if max_items and len(pairs) >= max_items:
                break
        return pairs

    def image_to_tensor(path: Path, device, dtype=torch.float16):
        img = Image.open(path).convert("RGB")
        img = center_crop_resize(img, RESOLUTION)
        arr = torch.from_numpy(__import__('numpy').array(img)).float() / 127.5 - 1.0
        return arr.permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=dtype)

    def get_sigmas(scheduler, timesteps, n_dim, dtype, device):
        sigmas = scheduler.sigmas.to(device=device, dtype=dtype)
        schedule_ts = scheduler.timesteps.to(device)
        step_indices = []
        for ts in timesteps:
            matches = (schedule_ts == ts).nonzero()
            step_indices.append(matches[0].item() if len(matches) else 0)
        sigma = sigmas[step_indices].flatten()
        while len(sigma.shape) < n_dim:
            sigma = sigma.unsqueeze(-1)
        return sigma

    def append_log(row: dict, log_path: Path):
        write_header = not log_path.exists()
        with log_path.open("a", encoding="utf-8", newline="") as fh:
            writer = _csv.DictWriter(fh, fieldnames=list(row.keys()))
            if write_header:
                writer.writeheader()
            writer.writerow(row)

    # ── Load model ───────────────────────────────────────────────────────────
    print("Loading SD3.5 pipeline ...")
    load_kwargs = {
        "torch_dtype": torch.float16,
        "use_safetensors": True,
    }
    if not USE_T5:
        load_kwargs.update({"text_encoder_3": None, "tokenizer_3": None})
    pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, **load_kwargs)
    pipe.scheduler.set_timesteps(1000, device=DEVICE)

    # Freeze everything except the transformer (which gets LoRA)
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    pipe.text_encoder_2.requires_grad_(False)
    if getattr(pipe, "text_encoder_3", None) is not None:
        pipe.text_encoder_3.requires_grad_(False)

    pipe.vae.to(DEVICE)
    pipe.transformer.to(DEVICE)
    pipe.transformer.set_attn_processor(AttnProcessor2_0())
    pipe.transformer.enable_gradient_checkpointing()

    # ── Attach LoRA ──────────────────────────────────────────────────────────
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["to_q", "to_k", "to_v", "to_out.0",
                        "add_q_proj", "add_k_proj", "add_v_proj"],
        init_lora_weights="gaussian",
    )
    pipe.transformer = get_peft_model(pipe.transformer, lora_config)
    pipe.transformer.print_trainable_parameters()
    pipe.transformer.train()

    # ── Optimizer ────────────────────────────────────────────────────────────
    optimizer = bnb.optim.AdamW8bit(
        filter(lambda p: p.requires_grad, pipe.transformer.parameters()),
        lr=LEARNING_RATE,
        weight_decay=1e-4,
    )

    # Cosine LR with warmup
    warmup_steps = int(MAX_TRAIN_STEPS * 0.05)
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, MAX_TRAIN_STEPS - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    scheduler_lr = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # ── Training loop ────────────────────────────────────────────────────────
    train_pairs = load_train_pairs(TRAIN_DATA_DIR, max_items=MAX_TRAIN_IMAGES)
    timesteps_sched = pipe.scheduler.timesteps.to(DEVICE)
    rng = random.Random(SEED)
    t0 = time.time()

    print(f"Manual training loop: {len(train_pairs)} images, {MAX_TRAIN_STEPS} steps")

    for step in range(MAX_TRAIN_STEPS):
        img_path, caption = rng.choice(train_pairs)

        pixel_values = image_to_tensor(img_path, device=DEVICE)

        with torch.no_grad():
            # Encode image → latents
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            scaling = getattr(pipe.vae.config, "scaling_factor", 1.0)
            shift   = getattr(pipe.vae.config, "shift_factor",   0.0)
            latents = (latents - shift) * scaling

            # Encode text
            prompt_embeds, _, pooled_embeds, _ = pipe.encode_prompt(
                prompt=caption,
                prompt_2=caption,
                prompt_3=caption if USE_T5 else None,
                device=DEVICE,
                num_images_per_prompt=1,
                do_classifier_free_guidance=False,
            )

        # Flow Matching: pick random timestep, add noise, predict target
        noise           = torch.randn_like(latents)
        ts_idx          = torch.randint(0, len(timesteps_sched), (1,), device=DEVICE)
        timestep        = timesteps_sched[ts_idx]
        sigmas          = get_sigmas(pipe.scheduler, timestep, latents.ndim,
                                     latents.dtype, DEVICE)
        noisy_latents   = sigmas * noise + (1.0 - sigmas) * latents
        target          = noise - latents  # flow matching target (v-prediction)

        model_pred = pipe.transformer(
            hidden_states=noisy_latents,
            timestep=timestep,
            encoder_hidden_states=prompt_embeds,
            pooled_projections=pooled_embeds,
            return_dict=False,
        )[0]

        loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
        loss.backward()

        # Gradient accumulation
        if (step + 1) % GRADIENT_ACCUM_STEPS == 0 or (step + 1) == MAX_TRAIN_STEPS:
            optimizer.step()
            scheduler_lr.step()
            optimizer.zero_grad(set_to_none=True)

        loss_val = float(loss.item())
        append_log({
            "step": step + 1,
            "loss": round(loss_val, 6),
            "lr": round(scheduler_lr.get_last_lr()[0], 8),
            "elapsed_sec": round(time.time() - t0, 1),
        }, TRAIN_LOG_CSV)

        if step == 0 or (step + 1) % 50 == 0:
            print(f"step {step+1:>5}/{MAX_TRAIN_STEPS} | loss={loss_val:.5f} | img={img_path.name}")

        # Checkpoint
        if (step + 1) % SAVE_STEPS == 0 or (step + 1) == MAX_TRAIN_STEPS:
            ckpt_dir = OUTPUT_DIR / f"checkpoint-{step+1}"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            pipe.transformer.save_pretrained(ckpt_dir / "transformer_peft")
            print(f"  Checkpoint saved: {ckpt_dir}")

    # Final save (standard diffusers format for load_lora_weights)
    from peft.utils import get_peft_model_state_dict
    import safetensors.torch
    lora_state = get_peft_model_state_dict(pipe.transformer)
    safetensors.torch.save_file(lora_state, str(OUTPUT_DIR / "pytorch_lora_weights.safetensors"))
    pipe.transformer.save_pretrained(OUTPUT_DIR / "transformer_peft")
    print(f"Manual training done. Weights saved to {OUTPUT_DIR}")

    del pipe
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("RUN_MANUAL_LOOP=False – skipping manual loop. Set to True if Section 10 fails.")

## 11. Plot Training Loss Curve

In [ ]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path

def plot_loss_from_csv(log_csv: Path, save_path: Path, window: int = 20):
    """Plot training loss from the manual-loop CSV log or the diffusers JSON log."""
    if not log_csv.exists():
        print(f"Log CSV not found: {log_csv}")
        print("If using train_dreambooth_lora_sd3.py, look for loss in its stdout output.")
        return

    steps, losses = [], []
    with log_csv.open("r", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            try:
                steps.append(int(row.get("step", row.get("Step", 0))))
                losses.append(float(row.get("loss", row.get("Loss", 0))))
            except (ValueError, KeyError):
                continue

    if not steps:
        print("No rows in log CSV.")
        return

    # Moving average
    smoothed = []
    for i in range(len(losses)):
        start = max(0, i - window + 1)
        smoothed.append(sum(losses[start:i+1]) / (i - start + 1))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps, losses,   alpha=0.35, color="steelblue", label="loss")
    ax.plot(steps, smoothed, color="darkorange",            label=f"smoothed (w={window})")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("SD3.5 LoRA Training Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    print(f"Loss curve saved to {save_path}")
    plt.show()

    print(f"Total steps logged : {len(steps)}")
    print(f"Final loss         : {losses[-1]:.5f}")
    print(f"Min loss           : {min(losses):.5f} at step {steps[losses.index(min(losses))]}")


plot_loss_from_csv(TRAIN_LOG_CSV, LOSS_CURVE_PNG)

## 12. List Saved Checkpoints

In [ ]:
import os

print(f"Contents of {OUTPUT_DIR}:")
for item in sorted(OUTPUT_DIR.iterdir()):
    if item.is_dir():
        size_mb = sum(f.stat().st_size for f in item.rglob("*") if f.is_file()) / 1e6
        print(f"  [DIR]  {item.name:40s}  {size_mb:.1f} MB")
else:
        size_mb = item.stat().st_size / 1e6
        print(f"  [FILE] {item.name:40s}  {size_mb:.1f} MB")

## 13. Sanity-Check Inference with Trained LoRA

We load the trained LoRA into a text-to-image pipeline on **cuda:1** (to keep training weights on cuda:0 if available)  
and generate sample images for each augmentation variant to visually verify the LoRA works.

> **Note:** If only one GPU is available, we fall back to cuda:0.

In [ ]:
import gc
import torch
import math
import matplotlib.pyplot as plt
from diffusers import StableDiffusion3Pipeline

# Choose device
n_gpus = torch.cuda.device_count()
INFER_DEVICE = "cuda:1" if n_gpus >= 2 else "cuda:0"
print(f"Inference device: {INFER_DEVICE}")


def load_lora_pipeline(model_id: str, lora_dir: Path, device: str, use_t5: bool = False):
    """Load SD3.5 pipeline with trained LoRA weights."""
    kwargs = {"torch_dtype": torch.float16, "use_safetensors": True}
    if not use_t5:
        kwargs.update({"text_encoder_3": None, "tokenizer_3": None})

    pipe = StableDiffusion3Pipeline.from_pretrained(model_id, **kwargs)

    # Try loading LoRA – supports both PEFT directory and safetensors file
    lora_safetensors = lora_dir / "pytorch_lora_weights.safetensors"
    peft_dir         = lora_dir / "transformer_peft"
    # Also search checkpoints
    checkpoints = sorted(lora_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))

    loaded = False
    if lora_safetensors.exists():
        try:
            pipe.load_lora_weights(str(lora_dir))
            print(f"Loaded LoRA safetensors from {lora_safetensors}")
            loaded = True
        except Exception as exc:
            print(f"load_lora_weights failed: {exc}")

    if not loaded and peft_dir.exists():
        try:
            pipe.transformer.load_adapter(str(peft_dir), adapter_name="citypersons_lora")
            pipe.transformer.set_adapter("citypersons_lora")
            print(f"Loaded PEFT adapter from {peft_dir}")
            loaded = True
        except Exception as exc:
            print(f"load_adapter failed: {exc}")

    if not loaded and checkpoints:
        latest_ckpt = checkpoints[-1]
        ckpt_peft   = latest_ckpt / "transformer_peft"
        if ckpt_peft.exists():
            try:
                pipe.transformer.load_adapter(str(ckpt_peft), adapter_name="citypersons_lora")
                pipe.transformer.set_adapter("citypersons_lora")
                print(f"Loaded PEFT adapter from checkpoint {latest_ckpt.name}")
                loaded = True
            except Exception as exc:
                print(f"Checkpoint load_adapter failed: {exc}")

    if not loaded:
        print("WARNING: No LoRA weights loaded. Using base model for sanity check.")

    pipe = pipe.to(device)
    if hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()
    return pipe, loaded


# ── Build inference prompts ────────────────────────────────────────────────────
SCENE_PREFIX = (
    "photorealistic CityPersons urban street scene, "
    "European city, dashcam perspective, "
).strip()

INFER_PROMPTS = {
    variant: f"{SCENE_PREFIX}, {vp}, high detail, natural lighting"
    for variant, vp in VARIANT_PROMPTS.items()
}

NEGATIVE_PROMPT = (
    "floating person, wrong scale, wrong perspective, bad anatomy, extra limbs, "
    "deformed face, warped body, unrealistic pose, no ground contact, "
    "text, watermark, cartoon, synthetic artifacts, severe blur"
)

# ── Run inference ─────────────────────────────────────────────────────────────
VARIANTS_TO_SAMPLE = list(INFER_PROMPTS.keys())[:4]  # sample 4 variants
INFER_STEPS        = 28
INFER_GUIDANCE     = 7.0
INFER_RESOLUTION   = 512

print("Loading pipeline for sanity-check inference ...")
infer_pipe, lora_loaded = load_lora_pipeline(MODEL_ID, OUTPUT_DIR, INFER_DEVICE, USE_T5)

generated = {}
for variant in VARIANTS_TO_SAMPLE:
    prompt   = INFER_PROMPTS[variant]
    out_path = INFERENCE_DIR / f"{variant}.png"
    try:
        gen = torch.Generator(device=INFER_DEVICE).manual_seed(SEED)
        result = infer_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=INFER_RESOLUTION,
            width=INFER_RESOLUTION,
            num_inference_steps=INFER_STEPS,
            guidance_scale=INFER_GUIDANCE,
            generator=gen,
        ).images[0]
        result.save(out_path)
        generated[variant] = result
        print(f"  Generated: {variant}")
    except Exception as exc:
        print(f"  FAILED {variant}: {exc}")

# ── Display results ────────────────────────────────────────────────────────────
if generated:
    cols = min(2, len(generated))
    rows = math.ceil(len(generated) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax, (variant, img) in zip(axes, generated.items()):
        ax.imshow(img)
        ax.set_title(variant.replace("_", " "), fontsize=9)
        ax.axis("off")
    for ax in axes[len(generated):]:
        ax.axis("off")
    lora_status = "with LoRA" if lora_loaded else "BASE MODEL (LoRA not loaded)"
    plt.suptitle(f"SD3.5 Inference Sanity Check – {lora_status}", fontsize=12)
    plt.tight_layout()
    plt.savefig(INFERENCE_DIR / "sanity_grid.png", dpi=120)
    plt.show()
    print(f"\nSanity check images saved to {INFERENCE_DIR}")

del infer_pipe
torch.cuda.empty_cache()
gc.collect()

## 14. Package & Export LoRA Weights

We create a final `.safetensors` snapshot that can be:
- Downloaded from Kaggle's output tab
- Uploaded as a Kaggle Dataset for the augmentation notebook to consume
- Shared on HuggingFace Hub

In [ ]:
import shutil
from pathlib import Path

EXPORT_DIR = Path("/kaggle/working/lora_export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Gather all weight files from OUTPUT_DIR
exported = []
for pattern in ["pytorch_lora_weights.safetensors", "*.safetensors", "*.bin"]:
    for src in OUTPUT_DIR.glob(pattern):
        dst = EXPORT_DIR / src.name
        shutil.copy2(src, dst)
        exported.append(dst)
        print(f"Copied: {src.name}  ({src.stat().st_size / 1e6:.1f} MB)")

# Copy PEFT dir if present
peft_src = OUTPUT_DIR / "transformer_peft"
if peft_src.exists():
    peft_dst = EXPORT_DIR / "transformer_peft"
    if peft_dst.exists():
        shutil.rmtree(peft_dst)
    shutil.copytree(peft_src, peft_dst)
    print(f"Copied PEFT directory: transformer_peft")
    exported.append(peft_dst)

# Copy latest checkpoint if no final weights
if not exported:
    checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"),
                         key=lambda p: int(p.name.split("-")[-1]))
    if checkpoints:
        latest = checkpoints[-1]
        dst = EXPORT_DIR / latest.name
        shutil.copytree(latest, dst)
        print(f"Copied latest checkpoint: {latest.name}")
        exported.append(dst)

# Write a README
readme = EXPORT_DIR / "README.md"
readme.write_text(
    f"# SD3.5 Medium LoRA – CityPersons Pedestrian Insertion\n\n"
    f"Model: `{MODEL_ID}`\n"
    f"Rank: {LORA_RANK}  |  Alpha: {LORA_ALPHA}\n"
    f"Steps: {MAX_TRAIN_STEPS}  |  Resolution: {RESOLUTION}\n"
    f"Training images: {n_prepared}\n\n"
    f"## Usage\n"
    f"```python\n"
    f"from diffusers import StableDiffusion3Img2ImgPipeline\n"
    f"pipe = StableDiffusion3Img2ImgPipeline.from_pretrained('{MODEL_ID}', ...)\n"
    f"pipe.load_lora_weights('/path/to/lora_export')\n"
    f"pipe.set_adapters('default_0', adapter_weights=0.8)\n"
    f"```\n",
    encoding="utf-8",
)

print(f"\nExported to {EXPORT_DIR}")
total_mb = sum(f.stat().st_size for f in EXPORT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Total export size: {total_mb:.1f} MB")

## 15. Integration with Augmentation Notebook

To use this LoRA in the **augmentation notebook** (`citypersons_sd35_lora_kaggle.ipynb`),  
set `LORA_DIR` to the exported directory:

```python
# In citypersons_sd35_lora_kaggle.ipynb → Section 4 Configuration:
LORA_DIR = Path("/kaggle/input/<your-lora-dataset>/lora_export")
```

The augmentation notebook's `build_img2img_pipeline()` will auto-detect and load the LoRA via `pipe.load_lora_weights()` or the PEFT adapter mechanism.

### Recommended LoRA scale for img2img
A LoRA scale of **0.7–0.9** typically gives the best balance between style adherence and content preservation:  
```python
pipe.set_adapters("default_0", adapter_weights=0.8)
```

## Summary

| Section | What it does |
|---|---|
| 1–2 | Install deps + download training script |
| 3–4 | Verify runtime + HF login |
| 5 | All hyperparams in one place |
| 6–7 | Prepare & preview training data |
| 8–9 | Configure accelerate + build command |
| 10 | `accelerate launch train_dreambooth_lora_sd3.py` |
| 10b | Fallback manual training loop |
| 11–12 | Loss curve + checkpoint listing |
| 13 | Sanity-check t2i inference with LoRA |
| 14–15 | Export + integration instructions |